# Kelmarsh Cross-Turbine and Cross-Year Validation

## Purpose

This notebook validates the Kelmarsh Wind Farm SCADA and status-event exports across all six turbines and all available years from 2016 through 2022.

Phase 1 used Turbine 1 in 2016 to understand the raw data and define a provisional data contract. This notebook tests whether those schemas, measurements, data-quality patterns, and investigation fields remain consistent across the complete dataset.

The analysis is designed to process files individually and create compact validation summaries without loading the complete raw dataset into memory.

## Main Questions

1. Are all expected turbine-year SCADA and status files available?
2. Do SCADA and status schemas remain consistent across turbines and years?
3. How do timestamp coverage, duplicates, and missing measurements vary?
4. Are the provisional investigation fields consistently available and meaningful?
5. Which fields should be required, optional, derived, or excluded from the final data contract?

## 1. Setup and File Discovery

This section imports the required libraries, locates the project root, and discovers all yearly SCADA and status-event files without loading their measurement data.

In [1]:
from pathlib import Path
import pandas as pd

In [2]:
current_directory = Path.cwd()
project_root = (
    current_directory.parent
    if current_directory.name == "notebooks"
    else current_directory
)

data_directory = project_root / "data" / "raw"

scada_files = sorted(data_directory.rglob("Turbine_Data_*.csv"))
status_files = sorted(data_directory.rglob("Status_*.csv"))


print(f"Project root detected: {project_root.name}")
print(f"Data directory exists: {data_directory.exists()}")
print(f"SCADA files found: {len(scada_files)}")
print(f"Status files found: {len(status_files)}")

Project root detected: wind-farm-investigation-copilot
Data directory exists: True
SCADA files found: 42
Status files found: 42


### File Discovery Findings

The expected seven years and six turbines produce 42 SCADA files. The discovery also found 42 status files, indicating that each expected turbine-year has both a SCADA export and a status-event export.

This establishes file-level availability. The following sections validate identifiers, schemas, timestamps, measurement coverage, and Status-event contents.

In [3]:
file_groups = [
    ("SCADA", scada_files),
    ("Status", status_files),
]

inventory_records = []

for file_type, files in file_groups:
    for file_path in files:
        inventory_records.append(
            {"file_type": file_type,
             "file_name": file_path.name,
             "year_folder": file_path.parent.name,
             "size_mb": file_path.stat().st_size / (1024 ** 2),
             "file_path": str(file_path)
             })

file_inventory = pd.DataFrame(inventory_records)
print(file_inventory.shape)
file_inventory[
    [
        "file_type",
        "file_name",
        "year_folder",
        "size_mb",
    ]
].head()

(84, 5)


,file_type,file_name,year_folder,size_mb
0,SCADA,Turbine_Data_Kelmarsh_1_2016-01-03_-_2017-01-0...,kelmarsh_2016,86.742239
1,SCADA,Turbine_Data_Kelmarsh_2_2016-01-03_-_2017-01-0...,kelmarsh_2016,86.755253
2,SCADA,Turbine_Data_Kelmarsh_3_2016-01-03_-_2017-01-0...,kelmarsh_2016,86.569201
3,SCADA,Turbine_Data_Kelmarsh_4_2016-01-03_-_2017-01-0...,kelmarsh_2016,86.431081
4,SCADA,Turbine_Data_Kelmarsh_5_2016-01-03_-_2017-01-0...,kelmarsh_2016,86.640081


In [4]:
inventory_by_year_type = file_inventory.groupby(["year_folder", "file_type"]).agg(
    file_count=("file_name", "count"),
    total_size_mb=("size_mb", "sum"),
    minimum_size_mb=("size_mb", "min"),
    maximum_size_mb=("size_mb", "max"),
).round(2)

inventory_by_year_type

file_count  total_size_mb  minimum_size_mb  \
year_folder   file_type                                               
kelmarsh_2016 SCADA               6         519.21            86.08   
              Status              6           1.63             0.21   
kelmarsh_2017 SCADA               6         642.31           106.69   
              Status              6           4.36             0.57   
kelmarsh_2018 SCADA               6         809.13           134.61   
              Status              6           6.16             0.83   
kelmarsh_2019 SCADA               6         857.54           140.44   
              Status              6           5.22             0.69   
kelmarsh_2020 SCADA               6        1172.91           194.66   
              Status              6           5.07             0.66   
kelmarsh_2021 SCADA               6        1183.99           196.28   
              Status              6           5.97             0.71   
kelmarsh_2022 SCADA               6        1209.44           200.44   
              Status              6           6.33             0.87   

                         maximum_size_mb  
year_folder   file_type                   
kelmarsh_2016 SCADA                86.76  
              Status                0.37  
kelmarsh_2017 SCADA               107.23  
              Status                1.02  
kelmarsh_2018 SCADA               135.13  
              Status                1.41  
kelmarsh_2019 SCADA               143.58  
              Status                1.21  
kelmarsh_2020 SCADA               195.98  
              Status                1.07  
kelmarsh_2021 SCADA               198.04  
              Status                1.28  
kelmarsh_2022 SCADA               202.23  
              Status                1.24

### File-Inventory Findings

Every year from 2016 through 2022 contains six SCADA files and six status files. The expected turbine-year file structure is therefore complete and consistent at the directory level.

SCADA file sizes increase substantially across the available years. File size alone cannot determine whether this is caused by additional rows, additional columns, different text representation, or a combination of these factors. Row counts and schemas must be inspected directly.

Status-file sizes vary between turbines within the same year. This may reflect different numbers of recorded events and different operational histories rather than incomplete data. File size is therefore treated as an anomaly indicator, not a data-quality conclusion.

In [5]:
identifier_pattern = r"Kelmarsh_(\d+)_(\d{4})"

parsed_identifiers = file_inventory["file_name"].str.extract(
    identifier_pattern
)

parsed_identifiers.columns = [
    "turbine_id",
    "file_year",
]

file_inventory[["turbine_id", "file_year"]] = parsed_identifiers

In [6]:
folder_year = file_inventory["year_folder"].str.extract(r"(\d{4})", expand=False)
#print(folder_year.isna().sum())
folder_year = folder_year.astype(int)
file_inventory["folder_year"] = folder_year

In [7]:
file_inventory["turbine_id"] = file_inventory["turbine_id"].astype(int)
file_inventory["file_year"] = file_inventory["file_year"].astype(int)

In [8]:
file_inventory["year_matches_folder"] = (
    file_inventory["file_year"]
    == file_inventory["folder_year"]
)

In [9]:
print(
    "Unique turbine IDs:",
    sorted(file_inventory["turbine_id"].unique()),
)

print(
    "Unique file years:",
    sorted(file_inventory["file_year"].unique()),
)

print(
    "Missing turbine identifiers:",
    file_inventory["turbine_id"].isna().sum(),
)

print(
    "Missing file-year identifiers:",
    file_inventory["file_year"].isna().sum(),
)

print(
    "Missing folder-year identifiers:",
    file_inventory["folder_year"].isna().sum(),
)

print(
    "Filename-folder year mismatches:",
    (~file_inventory["year_matches_folder"]).sum(),
)

Unique turbine IDs: [1, 2, 3, 4, 5, 6]
Unique file years: [2016, 2017, 2018, 2019, 2020, 2021, 2022]
Missing turbine identifiers: 0
Missing file-year identifiers: 0
Missing folder-year identifiers: 0
Filename-folder year mismatches: 0


In [10]:
turbine_year_file_counts = (
    file_inventory
    .groupby(
        [
            "file_year",
            "turbine_id",
            "file_type",
        ]
    )
    .size()
    .rename("file_count")
    .reset_index()
)

print(
    "Unique turbine-year-file-type combinations:",
    len(turbine_year_file_counts),
)

print("\nFrequency of file counts:")
print(
    turbine_year_file_counts["file_count"]
    .value_counts()
    .sort_index()
)

invalid_combination_count = (
    turbine_year_file_counts["file_count"] != 1
).sum()

print(
    "\nInvalid combinations:",
    invalid_combination_count,
)

Unique turbine-year-file-type combinations: 84

Frequency of file counts:
file_count
1    84
Name: count, dtype: int64

Invalid combinations: 0


### Identifier and Completeness Findings

Turbine and year identifiers were successfully extracted from all 84 filenames. The inventory contains turbine IDs 1–6 and years 2016–2022, with no missing identifiers.

Every filename year matches its containing year folder. Grouping by year, turbine, and file type produced all 84 expected combinations, and every combination contains exactly one file.

The acquisition is therefore complete at the file level: no expected SCADA or Status file is missing or repeated within a turbine-year-file-type combination. The following sections validate the contents of those files.

## 2. Validate Schemas Across Turbines and Years

We load only the CSV headers to compare column availability across every turbine and year. This avoids reading the full 8.4 GB dataset while identifying schema changes that affect the final data contract.

In [11]:
reference_scada_columns = pd.read_csv(scada_files[0], skiprows=9, nrows=0).columns.tolist()
reference_status_columns = pd.read_csv(status_files[0], skiprows=9, nrows=0).columns.tolist()

reference_schemas = {
    "SCADA": reference_scada_columns,
    "Status": reference_status_columns,
}

print("Reference SCADA columns:", len(reference_scada_columns))
print("Reference status columns:", len(reference_status_columns))

schema_records = []

for row in file_inventory.itertuples(index=False):
    current_columns = pd.read_csv(
        row.file_path,
        skiprows=9,
        nrows=0,
    ).columns.tolist()

    reference_columns = reference_schemas[row.file_type]

    schema_matches_reference = (
        current_columns == reference_columns
    )

    schema_records.append(
        {
            "file_type": row.file_type,
            "file_year": row.file_year,
            "turbine_id": row.turbine_id,
            "column_count": len(current_columns),
            "schema_matches_reference": schema_matches_reference,
            "column_names": tuple(current_columns),
        }
    )

schema_summary = pd.DataFrame(schema_records)
print("Schema summary shape: ",schema_summary.shape)
schema_summary.head()


Reference SCADA columns: 299
Reference status columns: 9
Schema summary shape:  (84, 6)


,file_type,file_year,turbine_id,column_count,schema_matches_reference,column_names
0,SCADA,2016,1,299,True,"(# Date and time, Wind speed (m/s), Wind speed..."
1,SCADA,2016,2,299,True,"(# Date and time, Wind speed (m/s), Wind speed..."
2,SCADA,2016,3,299,True,"(# Date and time, Wind speed (m/s), Wind speed..."
3,SCADA,2016,4,299,True,"(# Date and time, Wind speed (m/s), Wind speed..."
4,SCADA,2016,5,299,True,"(# Date and time, Wind speed (m/s), Wind speed..."


In [12]:
schema_by_year_type = schema_summary.groupby(["file_year", "file_type"]).agg(
    files=("turbine_id" ,"size"),
    column_counts=(
        "column_count",
        lambda values: sorted(values.unique()),
    ),
    unique_schema_count=("column_names", "nunique"),
    reference_matches=("schema_matches_reference", "sum"),

).reset_index()

schema_by_year_type

,file_year,file_type,files,column_counts,unique_schema_count,reference_matches
0,2016,SCADA,6,[299],1,6
1,2016,Status,6,[9],1,6
2,2017,SCADA,6,[299],1,6
3,2017,Status,6,[9],1,6
4,2018,SCADA,6,[299],1,6
5,2018,Status,6,[9],1,6
6,2019,SCADA,6,[299],1,6
7,2019,Status,6,[9],1,6
8,2020,SCADA,6,[299],1,6
9,2020,Status,6,[9],1,6


In [13]:
scada_2021_mask = (
    (schema_summary["file_type"] == "SCADA")
    & (schema_summary["file_year"] == 2021)
)

scada_2021_columns = schema_summary.loc[
    scada_2021_mask,
    "column_names",
].iloc[0]

added_scada_columns = sorted(
    set(scada_2021_columns) - set(reference_scada_columns)
)

removed_scada_columns = sorted(
    set(reference_scada_columns) - set(scada_2021_columns)
)

print("Added SCADA columns:", len(added_scada_columns))
for column in added_scada_columns:
    print(f"  + {column}")

print("\nRemoved SCADA columns:", len(removed_scada_columns))
for column in removed_scada_columns:
    print(f"  - {column}")


Added SCADA columns: 4
  + Investment Performance Ratio
  + Manufacturer Potential Power (SCADA) (kW)
  + Operating Performance Ratio
  + Potential Power Energy Budget (kW)

Removed SCADA columns: 0


In [14]:
status_2021_mask = (
    (schema_summary["file_type"] == "Status")
    & (schema_summary["file_year"] == 2021)
)

status_2021_columns = schema_summary.loc[
    status_2021_mask,
    "column_names",
].iloc[0]

added_status_columns = sorted(
    set(status_2021_columns) - set(reference_status_columns)
)

removed_status_columns = sorted(
    set(reference_status_columns) - set(status_2021_columns)
)

print("Added Status columns:", len(added_status_columns))
for column in added_status_columns:
    print(f"  + {column}")

print("\nRemoved Status columns:", len(removed_status_columns))
for column in removed_status_columns:
    print(f"  - {column}")

Added Status columns: 2
  + Custom contract category
  + Global contract category

Removed Status columns: 0


In [15]:
scada_2022_mask = (
    (schema_summary["file_type"] == "SCADA")
    & (schema_summary["file_year"] == 2022)
)

scada_2022_columns = schema_summary.loc[
    scada_2022_mask,
    "column_names",
].iloc[0]

status_2022_mask = (
    (schema_summary["file_type"] == "Status")
    & (schema_summary["file_year"] == 2022)
)

status_2022_columns = schema_summary.loc[
    status_2022_mask,
    "column_names",
].iloc[0]

print(
    "2021 and 2022 SCADA schemas identical:",
    scada_2021_columns == scada_2022_columns,
)

print(
    "2021 and 2022 Status schemas identical:",
    status_2021_columns == status_2022_columns,
)

2021 and 2022 SCADA schemas identical: True
2021 and 2022 Status schemas identical: True


### Schema-validation findings

All six turbines use an identical schema within each year and file type. Schema changes therefore occur consistently across the dataset rather than affecting isolated turbine files.

The SCADA schema contains 299 columns from 2016 through 2020 and expands to 303 columns in 2021. The four additions describe manufacturer potential power, energy-budget potential power, and investment and operating performance ratios. No original SCADA columns are removed. The Status schema similarly expands from 9 to 11 columns in 2021 through the addition of `Custom contract category` and `Global contract category`, with no original Status columns removed.

The 2021 and 2022 schemas are identical for both SCADA and Status data. The original 299 SCADA columns and 9 Status columns form the stable shared schema across 2016–2022.

Later validation showed that `Investment Performance Ratio` and `Potential Power Energy Budget (kW)` can be preserved as optional later-year fields, while `Operating Performance Ratio` is optional and nullable. `Manufacturer Potential Power (SCADA) (kW)` is excluded because it is entirely empty.

For Status data, `Custom contract category` is entirely empty and `Global contract category` is very sparse. These additions do not replace the older service-contract category and remain optional supporting fields.

## 3. Validate SCADA Timestamp Coverage

This section validates the timestamp structure of all 42 SCADA files before combining their measurements. Each file is processed independently using only its timestamp column to limit memory usage.

The checks establish the observed row count and date range for every turbine-year and identify missing, duplicated, or incorrectly ordered timestamps. Expected ten-minute coverage will be calculated only after the observed timestamp boundaries have been validated.

### 3.1 Observed timestamp integrity

In [16]:
timestamp_records = []
scada_inventory = file_inventory[file_inventory.file_type == "SCADA"]
for row in scada_inventory.itertuples(index=False):
    timestamp_data = pd.read_csv(
        row.file_path,
        skiprows=9,
        usecols=["# Date and time"],
        parse_dates=["# Date and time"]
    )
    timestamps = timestamp_data["# Date and time"]
    intervals = timestamps.diff()
    intervals = intervals.dropna()
    expected_interval = pd.Timedelta(minutes=10)
    irregular_interval_count = (intervals != expected_interval).sum()
    timestamp_records.append(
        {
            "file_year": row.file_year,
            "turbine_id": row.turbine_id,
            "row_count" : len(timestamps),
            "first_timestamp": timestamps.min(),
            "last_timestamp": timestamps.max(),
            "missing_timestamps": timestamps.isna().sum(),
            "duplicate_timestamps": timestamps.duplicated().sum(),
            "irregular_intervals": irregular_interval_count,
            "timestamps_in_order": timestamps.is_monotonic_increasing
        }
    )

timestamp_summary = pd.DataFrame(timestamp_records)
timestamp_summary.head()

,file_year,turbine_id,row_count,first_timestamp,last_timestamp,missing_timestamps,duplicate_timestamps,irregular_intervals,timestamps_in_order
0,2016,1,52416,2016-01-03,2016-12-31 23:50:00,0,0,0,True
1,2016,2,52416,2016-01-03,2016-12-31 23:50:00,0,0,0,True
2,2016,3,52416,2016-01-03,2016-12-31 23:50:00,0,0,0,True
3,2016,4,52416,2016-01-03,2016-12-31 23:50:00,0,0,0,True
4,2016,5,52416,2016-01-03,2016-12-31 23:50:00,0,0,0,True


In [17]:
timestamp_summary_by_year = timestamp_summary.groupby(["file_year"]).agg(
    files=("turbine_id", "count"),
    row_counts=("row_count", lambda x: sorted(x.unique())),
    first_timestamp_min=('first_timestamp', "min"),
    first_timestamp_max=('first_timestamp', "max"),
    last_timestamp_min=('last_timestamp', "min"),
    last_timestamp_max=('last_timestamp', "max"),
    missing_timestamps=("missing_timestamps", "sum"),
    duplicate_timestamps=("duplicate_timestamps", "sum"),
    irregular_intervals=("irregular_intervals", "sum"),
    all_timestamps_in_order=('timestamps_in_order', "all"),
).reset_index()

timestamp_summary_by_year

,file_year,files,row_counts,first_timestamp_min,first_timestamp_max,last_timestamp_min,last_timestamp_max,missing_timestamps,duplicate_timestamps,irregular_intervals,all_timestamps_in_order
0,2016,6,[52416],2016-01-03,2016-01-03,2016-12-31 23:50:00,2016-12-31 23:50:00,0,0,0,True
1,2017,6,[52560],2017-01-01,2017-01-01,2017-12-31 23:50:00,2017-12-31 23:50:00,0,0,0,True
2,2018,6,[52560],2018-01-01,2018-01-01,2018-12-31 23:50:00,2018-12-31 23:50:00,0,0,0,True
3,2019,6,[52560],2019-01-01,2019-01-01,2019-12-31 23:50:00,2019-12-31 23:50:00,0,0,0,True
4,2020,6,[52704],2020-01-01,2020-01-01,2020-12-31 23:50:00,2020-12-31 23:50:00,0,0,0,True
5,2021,6,[52560],2021-01-01,2021-01-01,2021-12-31 23:50:00,2021-12-31 23:50:00,0,0,0,True
6,2022,6,[52560],2022-01-01,2022-01-01,2022-12-31 23:50:00,2022-12-31 23:50:00,0,0,0,True


In [18]:
def expected_intervals_for_year(year):
    start = pd.Timestamp(year=year, month=1, day=1)
    end = pd.Timestamp(year=year+1, month=1, day=1)

    expected_timestamp = pd.date_range(start, end, freq="10min", inclusive="left")
    return len(expected_timestamp)

timestamp_summary["expected_row_count"] = timestamp_summary["file_year"].apply(expected_intervals_for_year)
timestamp_summary["absent_timestamp_count"] = timestamp_summary["expected_row_count"] - timestamp_summary["row_count"]
timestamp_summary["timestamp_coverage_percentage"] = timestamp_summary["row_count"] / timestamp_summary["expected_row_count"] * 100

coverage_summary = timestamp_summary[
    [
        "file_year",
        "row_count",
        "expected_row_count",
        "absent_timestamp_count",
        "timestamp_coverage_percentage",
    ]
].drop_duplicates()

coverage_summary


,file_year,row_count,expected_row_count,absent_timestamp_count,timestamp_coverage_percentage
0,2016,52416,52704,288,99.453552
6,2017,52560,52560,0,100.000000
12,2018,52560,52560,0,100.000000
18,2019,52560,52560,0,100.000000
24,2020,52704,52704,0,100.000000
30,2021,52560,52560,0,100.000000
36,2022,52560,52560,0,100.000000


### Timestamp-coverage findings

All 42 SCADA files contain ordered, duplicate-free timestamps with a continuous ten-minute interval between their first and last observations. Within each year, all six turbines have identical row counts and timestamp boundaries.

The 2016 files begin on January 3 and contain 52,416 of the 52,704 timestamps expected for a leap year. Therefore, January 1–2 are absent for every turbine, resulting in 99.45% calendar coverage. All SCADA files from 2017 through 2022 provide complete calendar-year timestamp coverage.

The missing 2016 period will be preserved and documented rather than imputed. Valid January 1–2 observations from later years will not be removed. Analyses involving January 2016 must account for its incomplete coverage.

## 4. Validate Core Investigation Fields

This section tests whether the measurements selected during the 2016 exploratory analysis remain available and meaningful across all turbines and years.

Each SCADA file is processed independently using only the candidate investigation fields. The resulting summaries will measure coverage, missingness, ranges, and meaningful-value availability without combining the complete raw dataset in memory.

### 4.1 Candidate-field availability

In [19]:
candidate_fields = [
    "Wind speed (m/s)",
    "Power (kW)",
    "Potential power default PC (kW)",
    "Data Availability",
    "Available Capacity for Production (kW)",
    "Lost Production to Downtime (kWh)",
    "Lost Production to Performance (kWh)",
    "Lost Production to Curtailment (Total) (kWh)"
]

scada_schema_summary = schema_summary[schema_summary["file_type"] == "SCADA"]
candidate_schema_records = []

for row in scada_schema_summary.itertuples(index=False):
    missing_fields = sorted(set(candidate_fields) - set(row.column_names))
    candidate_schema_records.append(
        {
            "file_year": row.file_year,
            "turbine_id": row.turbine_id,
            "candidate_field_count": len(candidate_fields),
            "missing_field_count": len(missing_fields),
            "missing_fields": missing_fields,
            "all_candidate_fields_available": len(missing_fields) == 0,
        }
    )

candidate_schema_summary = pd.DataFrame(candidate_schema_records)
candidate_schema_summary

,file_year,turbine_id,candidate_field_count,missing_field_count,missing_fields,all_candidate_fields_available
0,2016,1,8,0,[],True
1,2016,2,8,0,[],True
2,2016,3,8,0,[],True
3,2016,4,8,0,[],True
4,2016,5,8,0,[],True
5,2016,6,8,0,[],True
6,2017,1,8,0,[],True
7,2017,2,8,0,[],True
8,2017,3,8,0,[],True
9,2017,4,8,0,[],True


### Candidate-field availability finding

All eight provisional investigation fields exist in every SCADA file across all six turbines and seven years. The same candidate structure can therefore be used throughout the dataset without year-specific column substitutions.

This validates column availability only. The completeness, variation, ranges, and meaningful-value coverage of these fields must still be assessed before they can be classified as required or optional inputs.

### 4.2 Candidate-field value coverage

Column presence does not guarantee usable measurements. This subsection profiles each candidate field in every turbine-year to measure non-missing coverage and identify entirely empty, sparsely populated, or constant fields.

In [20]:
candidate_value_records = []

for row in scada_inventory.itertuples(index=False):
    candidate_data = pd.read_csv(row.file_path, skiprows=9, usecols=candidate_fields)
    for field in candidate_fields:
        field_data = candidate_data[field]
        candidate_value_records.append(
            {
                "file_year": row.file_year,
                "turbine_id": row.turbine_id,
                "field": field,
                "row_count": len(field_data),
                "non_missing_count": field_data.notna().sum(),
                "missing_count": field_data.isna().sum(),
                "coverage_percentage": field_data.notna().mean() * 100,
                "unique_value_count": field_data.nunique(),
                "nonzero_count": ((field_data != 0) & field_data.notna()).sum(),
                "minimum": field_data.min(),
                "maximum": field_data.max(),
            }
        )

candidate_value_summary = pd.DataFrame(candidate_value_records)
candidate_value_summary.head()

,file_year,turbine_id,field,row_count,non_missing_count,missing_count,coverage_percentage,unique_value_count,nonzero_count,minimum,maximum
0,2016,1,Wind speed (m/s),52416,48485,3931,92.500382,35881,48485,0.114947,22.91000
1,2016,1,Power (kW),52416,48485,3931,92.500382,44625,48484,-16.652830,2072.45459
2,2016,1,Potential power default PC (kW),52416,48485,3931,92.500382,31200,43003,0.000000,2050.00000
3,2016,1,Data Availability,52416,52416,0,100.000000,2,48485,0.000000,1.00000
4,2016,1,Available Capacity for Production (kW),52416,50325,2091,96.010760,138,46060,0.000000,2050.00000


In [21]:
candidate_coverage_by_field = candidate_value_summary.groupby("field").agg(
    files=("turbine_id", "size"),
    minimum_coverage_percentage=('coverage_percentage', "min"),
    median_coverage_percentage=('coverage_percentage', "median"),
    maximum_coverage_percentage=('coverage_percentage', "max"),
    entirely_empty_files=(
    "non_missing_count",
    lambda values: (values == 0).sum(),
    ),
    constant_files=("unique_value_count",lambda values: (values <= 1).sum())
).reset_index()

candidate_coverage_by_field = candidate_coverage_by_field.sort_values("minimum_coverage_percentage" ,ascending=False)

candidate_coverage_by_field


,field,files,minimum_coverage_percentage,median_coverage_percentage,maximum_coverage_percentage,entirely_empty_files,constant_files
1,Data Availability,42,100.000000,100.000000,100.000000,0,0
0,Available Capacity for Production (kW),42,90.050748,99.601408,99.992390,0,0
2,Lost Production to Curtailment (Total) (kWh),42,89.833257,99.988584,100.000000,0,36
3,Lost Production to Downtime (kWh),42,89.659646,99.601408,99.992390,0,0
4,Lost Production to Performance (kWh),42,88.896520,99.131944,99.990487,0,0
5,Potential power default PC (kW),42,88.896520,98.991629,99.897260,0,0
6,Power (kW),42,88.896520,99.010654,99.897260,0,0
7,Wind speed (m/s),42,88.896520,98.991629,99.897260,0,0


In [22]:
curtailment_value_summary = candidate_value_summary[
    candidate_value_summary["field"]
    == "Lost Production to Curtailment (Total) (kWh)"
].copy()

curtailment_value_summary = curtailment_value_summary[
    [
        "file_year",
        "turbine_id",
        "coverage_percentage",
        "non_missing_count",
        "unique_value_count",
        "nonzero_count",
        "minimum",
        "maximum",
    ]
].sort_values(
    by=["file_year", "turbine_id"]
).reset_index(drop=True)

curtailment_value_summary

,file_year,turbine_id,coverage_percentage,non_missing_count,unique_value_count,nonzero_count,minimum,maximum
0,2016,1,93.667964,49097,1,0,0.0,0.000000
1,2016,2,93.933150,49236,1,0,0.0,0.000000
2,2016,3,92.067308,48258,1,0,0.0,0.000000
3,2016,4,90.069826,47211,1,0,0.0,0.000000
4,2016,5,93.067002,48782,1,0,0.0,0.000000
5,2016,6,89.833257,47087,1,0,0.0,0.000000
6,2017,1,99.739346,52423,1,0,0.0,0.000000
7,2017,2,99.739346,52423,1,0,0.0,0.000000
8,2017,3,99.739346,52423,1,0,0.0,0.000000
9,2017,4,99.739346,52423,1,0,0.0,0.000000


#### Curtailment-field finding

`Lost Production to Curtailment (Total) (kWh)` exists in every turbine-year file, but its usefulness changes over time. All available values are zero for every turbine from 2016 through 2021. In 2022, all six turbines contain varying nonzero curtailment-loss values.

The earlier zero values do not prove that no curtailment occurred, because the metric may not have been populated meaningfully during those years. The field will therefore be treated as optional, year-specific supporting evidence rather than a required investigation input.

In [23]:
lowest_coverage_indices = candidate_value_summary.groupby("field")["coverage_percentage"].idxmin()

lowest_coverage_by_field = candidate_value_summary.loc[lowest_coverage_indices]
lowest_coverage_by_field = lowest_coverage_by_field[[
    "field",
    "file_year",
    "turbine_id",
    "row_count",
    "non_missing_count",
    "missing_count",
    "coverage_percentage",
]]

lowest_coverage_by_field = lowest_coverage_by_field.sort_values("coverage_percentage" ,ascending=True).reset_index(drop=True)

lowest_coverage_by_field

,field,file_year,turbine_id,row_count,non_missing_count,missing_count,coverage_percentage
0,Lost Production to Performance (kWh),2016,6,52416,46596,5820,88.896520
1,Potential power default PC (kW),2016,6,52416,46596,5820,88.896520
2,Power (kW),2016,6,52416,46596,5820,88.896520
3,Wind speed (m/s),2016,6,52416,46596,5820,88.896520
4,Lost Production to Downtime (kWh),2016,6,52416,46996,5420,89.659646
5,Lost Production to Curtailment (Total) (kWh),2016,6,52416,47087,5329,89.833257
6,Available Capacity for Production (kW),2016,6,52416,47201,5215,90.050748
7,Data Availability,2016,1,52416,52416,0,100.000000


#### Shared core-field missingness

In [24]:
core_fields = [
    "Wind speed (m/s)",
    "Power (kW)",
    "Potential power default PC (kW)",
]

core_missingness_records = []
for row in scada_inventory.itertuples(index=False):
    core_data = pd.read_csv(row.file_path, skiprows=9, usecols=core_fields)
    all_core_missing = core_data.isna().all(axis=1)
    any_core_missing = core_data.isna().any(axis=1)

    core_missingness_records.append(
        {
            "file_year": row.file_year,
            "turbine_id": row.turbine_id,
            "row_count": len(core_data),
            "all_core_missing_count": all_core_missing.sum(),
            "any_core_missing_count": any_core_missing.sum(),
            "partially_missing_count": any_core_missing.sum() - all_core_missing.sum(),
        }
    )

core_missingness_summary = pd.DataFrame(core_missingness_records)
core_missingness_summary.head()

,file_year,turbine_id,row_count,all_core_missing_count,any_core_missing_count,partially_missing_count
0,2016,1,52416,3931,3931,0
1,2016,2,52416,3933,3933,0
2,2016,3,52416,4924,5187,263
3,2016,4,52416,5534,5534,0
4,2016,5,52416,4661,4661,0


In [25]:
partial_missingness_exceptions = core_missingness_summary[core_missingness_summary["partially_missing_count"] > 0]

partial_missingness_exceptions.groupby(["file_year", "turbine_id"])
partial_missingness_exceptions

,file_year,turbine_id,row_count,all_core_missing_count,any_core_missing_count,partially_missing_count
2,2016,3,52416,4924,5187,263
7,2017,2,52560,355,515,160
10,2017,5,52560,489,538,49
14,2018,3,52560,1949,2136,187
16,2018,5,52560,1806,1990,184
19,2019,2,52560,71,72,1
22,2019,5,52560,81,82,1
25,2020,2,52704,451,452,1
26,2020,3,52704,494,495,1
28,2020,5,52704,455,456,1


#### Candidate-field value-coverage conclusions

None of the eight candidate fields is entirely empty in any turbine-year file. `Data Availability` is the only field with 100% coverage across all 42 files. The remaining fields generally have high coverage, with median turbine-year coverage between approximately 98.99% and 99.99%.

The lowest coverage is concentrated in 2016 Turbine 6 rather than being widespread across later years. In this file, wind speed, actual power, default potential power, and performance-loss measurements each contain 5,820 missing values, corresponding to 88.90% coverage. Available capacity, downtime loss, and curtailment loss have slightly higher minimum coverage.

The curtailment-loss field requires year-specific treatment. It contains only zero values for all turbines from 2016 through 2021 and contains varying nonzero values only in 2022. It is therefore retained as optional supporting evidence rather than treated as a required all-year input.

Wind speed, actual power, and default potential power are frequently missing together, but their missingness is not identical in every file. Fourteen turbine-year files contain partially missing core rows, representing 879 observations where at least one—but not all—of the three values is missing.

Raw observations will be preserved. The analysis-ready dataset will require all three core measurements and exclude rows where any required core value is missing rather than assuming a universal shared-missingness pattern or automatically imputing these measurements.

### 4.3 Value ranges, rated capacity, and loss evidence

This subsection validates whether candidate-field values remain within plausible and consistent ranges across turbine-years. It also tests the 2,050 kW rated-capacity assumption and identifies which loss fields contain meaningful nonzero evidence.

In [26]:
candidate_range_summary = candidate_value_summary.groupby("field").agg(
    files=("turbine_id", "size"),
    overall_minimum=("minimum","min"),
    overall_maximum=("maximum","max"),
    files_with_negative_values=(
    "minimum",
    lambda values: (values < 0).sum(),
    ),
    files_with_nonzero_values=(
    "nonzero_count",
    lambda values: (values > 0).sum(),
    ),
    total_nonzero_observations=("nonzero_count", "sum"),
).reset_index()

candidate_range_summary

,field,files,overall_minimum,overall_maximum,files_with_negative_values,files_with_nonzero_values,total_nonzero_observations
0,Available Capacity for Production (kW),42,0.000000,2050.000000,0,42,2107146
1,Data Availability,42,0.000000,1.000000,0,42,2154565
2,Lost Production to Curtailment (Total) (kWh),42,0.000000,159.588926,0,6,1585
3,Lost Production to Downtime (kWh),42,0.000000,343.430909,0,42,69576
4,Lost Production to Performance (kWh),42,-187.597354,341.666667,42,42,1897856
5,Potential power default PC (kW),42,0.000000,2050.000000,0,42,1887828
6,Power (kW),42,-21.170280,2086.926514,42,42,2155377
7,Wind speed (m/s),42,0.000000,26.389881,0,42,2154528


#### Rated-capacity validation

Observed available capacity and default potential power are capped at 2,050 kW, while actual power occasionally exceeds this value slightly. The published turbine static metadata is inspected to confirm whether 2,050 kW is the rated capacity for all six turbines.

In [27]:
static_data_path = data_directory / "metadata" / "Kelmarsh_WT_static.csv"
turbine_static_data = pd.read_csv(static_data_path)
print("Shape:", turbine_static_data.shape)
print("Columns:", turbine_static_data.columns.tolist())

turbine_static_data

Shape: (6, 14)
Columns: ['Wind Farm', 'Title', 'Alternative Title', 'Identity', 'Manufacturer', 'Model', 'Rated power (kW)', 'Hub Height (m)', 'Rotor Diameter (m)', 'Latitude', 'Longitude', 'Elevation (m)', 'Country', 'Commercial Operations Date']


,Wind Farm,Title,Alternative Title,Identity,Manufacturer,Model,Rated power (kW),Hub Height (m),Rotor Diameter (m),Latitude,Longitude,Elevation (m),Country,Commercial Operations Date
0,Kelmarsh,Kelmarsh 1,KWF1,SEN 93420,Senvion,MM92,2050,78.5,92,52.400604,-0.947133,145.598,UK,15/04/2016
1,Kelmarsh,Kelmarsh 2,KWF2,SEN 93421,Senvion,MM92,2050,78.5,92,52.402551,-0.949527,156.577,UK,15/04/2016
2,Kelmarsh,Kelmarsh 3,KWF3,SEN 93422,Senvion,MM92,2050,68.5,92,52.403834,-0.944190,153.477,UK,15/04/2016
3,Kelmarsh,Kelmarsh 4,KWF4,SEN 93423,Senvion,MM92,2050,78.5,92,52.398781,-0.941150,146.313,UK,15/04/2016
4,Kelmarsh,Kelmarsh 5,KWF5,SEN 93424,Senvion,MM92,2050,78.5,92,52.402308,-0.940537,142.901,UK,15/04/2016
5,Kelmarsh,Kelmarsh 6,KWF6,SEN 93425,Senvion,MM92,2050,68.5,92,52.400687,-0.936093,135.039,UK,15/04/2016


#### Rated-capacity finding

The supplied static metadata identifies all six turbines as Senvion MM92 units with a rated power of 2,050 kW. This value is consistent with the observed maximum available capacity and default potential power across the SCADA files.

The project can therefore use 2,050 kW as the rated-capacity reference for every turbine. Actual power occasionally exceeds the nominal rating slightly; these observations will not be removed automatically because short exceedances may reflect normal measurement or operational behaviour.

The metadata also shows that Turbines 3 and 6 have lower hub heights than the other turbines. This information will be preserved as optional static context for later cross-turbine modelling.

In [28]:
actual_power_file_summary = candidate_value_summary[candidate_value_summary["field"] == "Power (kW)"].copy()

actual_power_file_summary["maximum_exceeds_rated_power"] = actual_power_file_summary["maximum"] > 2050

print("Number of files exceeding rated power: ", actual_power_file_summary["maximum_exceeds_rated_power"].sum())
print("Number of files not exceeding rated power: ", (actual_power_file_summary["maximum_exceeds_rated_power"] == False).sum())

print("largest observed power: ", actual_power_file_summary["maximum"].max())


Number of files exceeding rated power:  42
Number of files not exceeding rated power:  0
largest observed power:  2086.92651367188


In [29]:
actual_power_file_summary["maximum_excess_kw"] = (
    actual_power_file_summary["maximum"] - 2050
)

actual_power_file_summary["maximum_excess_percentage"] = (
    actual_power_file_summary["maximum_excess_kw"] / 2050 * 100
)

actual_power_file_summary[
    ["maximum_excess_kw", "maximum_excess_percentage"]
].describe()

,maximum_excess_kw,maximum_excess_percentage
count,42.000000,42.000000
mean,27.160286,1.324892
std,4.762294,0.232307
min,13.664062,0.666540
25%,25.235147,1.230983
50%,27.676935,1.350094
75%,30.327637,1.479397
max,36.926514,1.801293


#### Value-range and loss-evidence conclusions

The static metadata confirms that all six Senvion MM92 turbines have a rated power of 2,050 kW. Observed available capacity and default potential power are capped at this value, supporting its use as the common rated-capacity reference.

Every turbine-year contains at least one actual-power measurement above the nominal rating. The maximum exceedance ranges from approximately 0.67% to 1.80%, with a median of 1.35%. Because this small exceedance is consistent across all 42 files, actual power will not be clipped automatically at 2,050 kW.

Negative actual-power values also occur in every turbine-year. They may reflect auxiliary consumption or operational measurement behaviour and will be preserved for interpretation with capacity and Status evidence rather than treated automatically as invalid.

Downtime-loss values are nonnegative and contain nonzero evidence in all 42 files. Performance-loss values range from approximately -187.60 to 341.67 kWh and include negative values in every turbine-year. This field must therefore be treated as a signed vendor performance metric rather than an always-positive loss quantity; negative values will not be removed automatically.

Curtailment loss remains optional year-specific evidence because meaningful nonzero values appear only in 2022. Wind-speed values range from 0 to approximately 26.39 m/s and show no negative values in the profiled data.

The static metadata also records different hub heights for Turbines 3 and 6. Hub height will be preserved as optional turbine context for later cross-turbine modelling.

## 5. Validate Status-Event Data Across Turbines and Years

Status-event records provide supporting evidence for explaining SCADA underperformance periods. This section validates whether event timestamps, durations, categories, and messages remain usable across all turbines and years.

Each Status file is processed independently. Open-ended records marked with `-` are preserved as start-only events rather than treated automatically as corrupted data.

### 5.1 Event timestamp and record-format integrity

In [30]:
status_inventory = file_inventory[file_inventory["file_type"] == "Status"]

status_validation_records = []
for row in status_inventory.itertuples(index=False):
    status_data = pd.read_csv(row.file_path, skiprows=9)
    start_raw = status_data["Timestamp start"]
    end_raw = status_data["Timestamp end"]
    duration_raw = status_data["Duration"]

    end_clean = end_raw.replace("-", pd.NA)
    duration_clean = duration_raw.replace("-", pd.NA)

    parsed_start = pd.to_datetime(start_raw, errors="coerce")
    parsed_end = pd.to_datetime(end_clean, errors="coerce")
    parsed_duration = pd.to_timedelta(duration_clean, errors="coerce")

    record_count = len(status_data)

    start_only_count = (end_raw == "-").sum()

    interval_count = end_clean.notna().sum()

    invalid_start_count = (
    start_raw.notna() & parsed_start.isna()
    ).sum()

    invalid_end_count = (
        end_clean.notna() & parsed_end.isna()
    ).sum()

    invalid_duration_count = (
        duration_clean.notna() & parsed_duration.isna()
    ).sum()

    end_before_start_count = (
        parsed_start.notna()
        & parsed_end.notna()
        & (parsed_end < parsed_start)
    ).sum()

    status_validation_records.append(
        {
            "file_year": row.file_year,
            "turbine_id": row.turbine_id,
            "record_count": record_count,
            "first_start": parsed_start.min(),
            "last_start": parsed_start.max(),
            "start_only_count": start_only_count,
            "interval_count": interval_count,
            "record_types_complete": (
                start_only_count + interval_count == record_count
            ),
            "invalid_start_count": invalid_start_count,
            "invalid_end_count": invalid_end_count,
            "invalid_duration_count": invalid_duration_count,
            "end_before_start_count": end_before_start_count,
        }
    )


In [31]:
status_validation_summary = pd.DataFrame(
    status_validation_records
)

status_validation_summary.head()

,file_year,turbine_id,record_count,first_start,last_start,start_only_count,interval_count,record_types_complete,invalid_start_count,invalid_end_count,invalid_duration_count,end_before_start_count
0,2016,1,2122,2016-01-14 19:28:03,2016-12-27 15:24:04,649,1473,True,0,0,0,0
1,2016,2,1790,2016-01-21 14:11:59,2016-12-27 15:27:25,550,1240,True,0,0,1,1
2,2016,3,2873,2016-01-27 15:56:48,2016-12-29 11:45:05,1000,1873,True,0,0,0,0
3,2016,4,1929,2016-02-04 18:49:54,2016-12-27 15:27:27,651,1278,True,0,0,0,0
4,2016,5,2116,2016-01-24 15:12:45,2016-12-29 12:23:20,719,1397,True,0,0,0,0


In [32]:
status_validation_exceptions = status_validation_summary[
    (status_validation_summary["record_types_complete"] == False) |
    (status_validation_summary["invalid_start_count"] > 0) |
    (status_validation_summary["invalid_end_count"] > 0) |
    (status_validation_summary["invalid_duration_count"] > 0) |
    (status_validation_summary["end_before_start_count"] > 0)
]

status_validation_exceptions = status_validation_exceptions.sort_values(["file_year", "turbine_id"])
status_validation_exceptions

,file_year,turbine_id,record_count,first_start,last_start,start_only_count,interval_count,record_types_complete,invalid_start_count,invalid_end_count,invalid_duration_count,end_before_start_count
1,2016,2,1790,2016-01-21 14:11:59,2016-12-27 15:27:25,550,1240,True,0,0,1,1


In [33]:
problem_file_mask = (
    (status_inventory["file_year"] == 2016)
    & (status_inventory["turbine_id"] == 2)
)

problem_status_path = status_inventory.loc[
    problem_file_mask,
    "file_path",
].iloc[0]

Path(problem_status_path).relative_to(project_root)

WindowsPath('data/raw/kelmarsh_2016/Status_Kelmarsh_2_2016-01-03_-_2017-01-01_229.csv')

In [34]:
problem_status_data = pd.read_csv(
    problem_status_path,
    skiprows=9,
)

problem_start = pd.to_datetime(
    problem_status_data["Timestamp start"],
    errors="coerce",
)

problem_end = pd.to_datetime(
    problem_status_data["Timestamp end"].replace("-", pd.NA),
    errors="coerce",
)

problem_duration = pd.to_timedelta(
    problem_status_data["Duration"].replace("-", pd.NA),
    errors="coerce",
)



In [35]:
# Preserve valid duration values while treating "-" as start-only.
problem_duration_clean = problem_status_data["Duration"].replace(
    "-",
    pd.NA,
)

# A supplied duration that failed timedelta parsing.
invalid_duration_mask = (
    problem_duration_clean.notna()
    & problem_duration.isna()
)

# A completed interval whose end occurs before its start.
reversed_interval_mask = (
    problem_start.notna()
    & problem_end.notna()
    & (problem_end < problem_start)
)

# Select a row if either validation problem occurs.
problem_record_mask = (
    invalid_duration_mask
    | reversed_interval_mask
)

problem_record = problem_status_data.loc[
    problem_record_mask,
    [
        "Timestamp start",
        "Timestamp end",
        "Duration",
        "Status",
        "Code",
        "Message",
        "IEC category",
    ],
]

problem_record

,Timestamp start,Timestamp end,Duration,Status,Code,Message,IEC category
1622,2016-10-10 13:43:49,2016-10-10 13:07:00,-01:-36:-49,Stop,20,Manual stop - on site,Scheduled Maintenance


#### Malformed status-interval finding

One record in the 2016 Turbine 2 Status data has an end timestamp earlier than its start timestamp and an invalid negative duration. Its valid descriptive evidence—start timestamp, status code, message, and IEC category—should be preserved. However, it must be flagged as an invalid interval, and its end timestamp and duration must not be used for interval-overlap calculations.

### 5.2 Closed-interval duration consistency

For valid closed intervals, the recorded duration should equal the difference between the end and start timestamps. This check determines whether the supplied duration can be trusted across turbine-years.

In [36]:
duration_consistency_records = []

for row in status_inventory.itertuples(index=False):
    status_data = pd.read_csv(
        row.file_path,
        skiprows=9,
    )

    parsed_start = pd.to_datetime(
        status_data["Timestamp start"],
        errors="coerce",
    )

    parsed_end = pd.to_datetime(
        status_data["Timestamp end"].replace("-", pd.NA),
        errors="coerce",
    )

    parsed_duration = pd.to_timedelta(
        status_data["Duration"].replace("-", pd.NA),
        errors="coerce",
    )

    valid_interval_mask = (
        parsed_start.notna()
        & parsed_end.notna()
        & parsed_duration.notna()
        & (parsed_end >= parsed_start)
    )

    calculated_duration = parsed_end - parsed_start

    duration_difference = (
        calculated_duration - parsed_duration
    ).abs()

    duration_mismatch_count = (
        valid_interval_mask
        & (duration_difference > pd.Timedelta(0))
    ).sum()

    duration_consistency_records.append(
        {
            "file_year": row.file_year,
            "turbine_id": row.turbine_id,
            "valid_interval_count": valid_interval_mask.sum(),
            "duration_mismatch_count": duration_mismatch_count,
        }
    )

duration_consistency_summary = pd.DataFrame(
    duration_consistency_records
)

print(
    "Valid closed intervals:",
    duration_consistency_summary["valid_interval_count"].sum(),
)

print(
    "Duration mismatches:",
    duration_consistency_summary["duration_mismatch_count"].sum(),
)

print(
    "Files containing mismatches:",
    (
        duration_consistency_summary["duration_mismatch_count"] > 0
    ).sum(),
)

Valid closed intervals: 63523
Duration mismatches: 0
Files containing mismatches: 0


#### Duration-consistency finding

Across all turbine-years, 63,523 valid closed Status intervals were assessed. Every recorded duration exactly matched the difference between its end and start timestamps.

Recorded durations can therefore be trusted for valid closed intervals. The single malformed 2016 Turbine 2 record does not invalidate the remaining intervals; it should be preserved as flagged supporting evidence but excluded from duration-based and interval-overlap calculations.

### 5.3 Status evidence-field availability

Valid timestamps are only useful when Status records also contain interpretable operational evidence. This section measures the availability and diversity of the stable status, code, message, service-category, and IEC-category fields across every turbine-year.

In [37]:
status_evidence_fields = [
    "Status",
    "Code",
    "Message",
    "Service contract category",
    "IEC category",
]

status_evidence_records = []

for row in status_inventory.itertuples(index=False):
    status_data = pd.read_csv(
        row.file_path,
        skiprows=9,
    )

    row_count = len(status_data)

    for field in status_evidence_fields:
        field_data = status_data[field]

        non_missing_count = field_data.notna().sum()

        coverage_percentage = (
            non_missing_count / row_count * 100
        )

        unique_value_count = field_data.nunique(
            dropna=True
        )

        status_evidence_records.append(
            {
                "file_year": row.file_year,
                "turbine_id": row.turbine_id,
                "field": field,
                "row_count": row_count,
                "non_missing_count": non_missing_count,
                "coverage_percentage": coverage_percentage,
                "unique_value_count": unique_value_count,
            }
        )

status_evidence_summary = pd.DataFrame(
    status_evidence_records
)

status_evidence_by_field = (
    status_evidence_summary
    .groupby("field")
    .agg(
        files=("field", "size"),
        minimum_coverage_percentage=(
            "coverage_percentage",
            "min",
        ),
        median_coverage_percentage=(
            "coverage_percentage",
            "median",
        ),
        maximum_coverage_percentage=(
            "coverage_percentage",
            "max",
        ),
        minimum_unique_values=(
            "unique_value_count",
            "min",
        ),
        maximum_unique_values=(
            "unique_value_count",
            "max",
        ),
    )
    .reset_index()
)

status_evidence_by_field

,field,files,minimum_coverage_percentage,median_coverage_percentage,maximum_coverage_percentage,minimum_unique_values,maximum_unique_values
0,Code,42,100.000000,100.000000,100.000000,51,81
1,IEC category,42,92.836946,99.296499,99.739540,6,8
2,Message,42,100.000000,100.000000,100.000000,51,81
3,Service contract category,42,18.274071,26.176510,99.121982,10,18
4,Status,42,100.000000,100.000000,100.000000,4,5


In [38]:
contract_category_fields = [
    "Service contract category",
    "Custom contract category",
    "Global contract category",
]

later_status_inventory = status_inventory[status_inventory["file_year"] >= 2021].copy()

contract_category_records = []
for row in later_status_inventory.itertuples(index=False):
    later_status_data = pd.read_csv(row.file_path, skiprows=9, usecols=contract_category_fields)

    later_row_count = len(later_status_data)
    for field in contract_category_fields:
        later_field_data = later_status_data[field]

        later_non_missing_count = later_field_data.notna().sum()

        later_coverage_percentage = (
            later_non_missing_count / later_row_count * 100
        )

        later_unique_value_count = later_field_data.nunique(
            dropna=True
        )

        contract_category_records.append(
            {
                "file_year": row.file_year,
                "turbine_id": row.turbine_id,
                "field": field,
                "row_count": later_row_count,
                "non_missing_count": later_non_missing_count,
                "coverage_percentage": later_coverage_percentage,
                "unique_value_count": later_unique_value_count,
            }
        )


contract_category_summary = pd.DataFrame(contract_category_records)

contract_category_by_field = contract_category_summary.groupby("field").agg(
    minimum_coverage_percentage=(
            "coverage_percentage",
            "min",
        ),
        median_coverage_percentage=(
            "coverage_percentage",
            "median",
        ),
        maximum_coverage_percentage=(
            "coverage_percentage",
            "max",
        ),
        minimum_unique_values=(
            "unique_value_count",
            "min",
        ),
        maximum_unique_values=(
            "unique_value_count",
            "max",
        ),
    ).reset_index()

contract_category_by_field

,field,minimum_coverage_percentage,median_coverage_percentage,maximum_coverage_percentage,minimum_unique_values,maximum_unique_values
0,Custom contract category,0.000000,0.000000,0.000000,0,0
1,Global contract category,0.833749,1.203882,1.862099,6,6
2,Service contract category,20.396527,26.130236,28.370663,11,16


#### Status evidence-field findings

`Status`, `Code`, and `Message` have complete coverage in all 42 Status files and can serve as required descriptive evidence fields in the cleaned Status table.

`IEC category` remains useful but nullable because its minimum turbine-year coverage is approximately 92.84%. Investigations may use it when available but must not depend on every event having an IEC classification.

`Service contract category` has strongly inconsistent coverage across years. It remains the most populated contract-category field in 2021–2022, with approximately 20.40–28.37% coverage, but it cannot be treated as a required event-level field.

The two fields introduced in 2021 do not replace the older service-contract category. `Custom contract category` is entirely empty, while `Global contract category` covers only approximately 0.83–1.86% of records. Contract-category fields should therefore be preserved as optional supporting evidence, while the investigation workflow relies primarily on the complete status, code, and message fields.

### 5.4 Exact duplicate Status records

Exact duplicate Status rows could cause the processed investigation data to count the same event evidence more than once. This section measures duplicate records within each source file before defining the cleaned-table rule.

In [39]:
status_duplicate_records = []

for row in status_inventory.itertuples(index=False):
    status_data = pd.read_csv(row.file_path, skiprows=9)
    status_duplicate_records.append(
        {
            "file_year": row.file_year,
            "turbine_id": row.turbine_id,
            "record_count": len(status_data),
            "duplicate_row_count": status_data.duplicated().sum(),
        }
    )

status_duplicate_summary = pd.DataFrame(status_duplicate_records)
status_duplicate_files = status_duplicate_summary[
    status_duplicate_summary["duplicate_row_count"] > 0
].sort_values(
    by="duplicate_row_count",
    ascending=False,
)


total_status_records = status_duplicate_summary[
    "record_count"
].sum()

total_duplicate_rows = status_duplicate_summary[
    "duplicate_row_count"
].sum()

files_with_duplicates = (
    status_duplicate_summary["duplicate_row_count"] > 0
).sum()

duplicate_percentage = (
    total_duplicate_rows / total_status_records * 100
)

print("Total Status records:", total_status_records)
print("Exact duplicate rows:", total_duplicate_rows)
print("Files containing duplicates:", files_with_duplicates)
print(
    "Duplicate percentage:",
    round(duplicate_percentage, 2),
)

status_duplicate_files

Total Status records: 385133
Exact duplicate rows: 573
Files containing duplicates: 31
Duplicate percentage: 0.15


,file_year,turbine_id,record_count,duplicate_row_count
37,2022,2,10739,69
34,2021,5,11008,68
35,2021,6,14033,51
32,2021,3,12262,45
30,2021,1,9997,42
5,2016,6,3189,40
4,2016,5,2116,35
33,2021,4,10311,32
31,2021,2,7717,26
2,2016,3,2873,22


#### Exact-duplicate finding

Across 385,133 Status records, 573 rows are redundant exact duplicates affecting 31 of the 42 files. They represent approximately 0.15% of all Status records, so duplication is limited but sufficiently widespread to require a consistent cleaning rule.

The raw exports will remain unchanged. In the cleaned Status table, one copy of each exact record will be retained while redundant identical copies are removed to prevent repeated evidence from affecting event counts or investigation summaries. A duplicate count or quality flag will preserve information about the original duplication.

## 6. Validate SCADA Fields Introduced in 2021

Four SCADA fields appear only in the 2021–2022 schema. Their coverage and value diversity must be validated before deciding whether they provide useful optional evidence or should be excluded from the cross-year data contract.

In [40]:
later_scada_fields = [
    "Investment Performance Ratio",
    "Manufacturer Potential Power (SCADA) (kW)",
    "Operating Performance Ratio",
    "Potential Power Energy Budget (kW)",
]

In [41]:
later_scada_inventory = scada_inventory[scada_inventory["file_year"] >= 2021].copy()

later_scada_field_records = []
for row in later_scada_inventory.itertuples(index=False):
    later_scada_data = pd.read_csv(row.file_path, skiprows=9, usecols=later_scada_fields)
    scada_row_count = len(later_scada_data)
    for field in later_scada_fields:
        later_scada_field_data = later_scada_data[field]
        later_scada_non_missing = later_scada_field_data.notna().sum()
        later_scada_coverage_percentage = later_scada_non_missing / scada_row_count * 100
        later_scada_unique_value_count = later_scada_field_data.nunique(dropna=True)
        min_later_scada_field_data = later_scada_field_data.min()
        max_later_scada_field_data = later_scada_field_data.max()

        later_scada_field_records.append(
            {
                "file_year": row.file_year,
                "turbine_id": row.turbine_id,
                "field": field,
                "row_count": scada_row_count,
                "non_missing_count": later_scada_non_missing,
                "coverage_percentage": later_scada_coverage_percentage,
                "unique_value_count": later_scada_unique_value_count,
                "minimum": min_later_scada_field_data,
                "maximum": max_later_scada_field_data,
            }
        )

later_scada_field_summary = pd.DataFrame(later_scada_field_records)

later_scada_fields_by_field = later_scada_field_summary.groupby("field").agg(
    minimum_coverage_percentage=(
            "coverage_percentage",
            "min",
        ),
        median_coverage_percentage=(
            "coverage_percentage",
            "median",
        ),
        maximum_coverage_percentage=(
            "coverage_percentage",
            "max",
        ),
        minimum_unique_values=(
            "unique_value_count",
            "min",
        ),
        maximum_unique_values=(
            "unique_value_count",
            "max",
        ),
    overall_min=("minimum", "min"),
    overall_max=("maximum", "max")
).reset_index()

later_scada_fields_by_field

,field,minimum_coverage_percentage,median_coverage_percentage,maximum_coverage_percentage,minimum_unique_values,maximum_unique_values,overall_min,overall_max
0,Investment Performance Ratio,100.000000,100.000000,100.000000,51241,52454,-0.041825,9.399810
1,Manufacturer Potential Power (SCADA) (kW),0.000000,0.000000,0.000000,0,0,NaN,NaN
2,Operating Performance Ratio,81.495434,84.370244,87.983257,42834,46243,-0.566277,19.532982
3,Potential Power Energy Budget (kW),100.000000,100.000000,100.000000,24,25,108.223751,842.712997


### Later-year SCADA field findings

The four SCADA fields introduced in 2021 cannot be required in the common 2016–2022 data contract because they are absent from all earlier years.

`Manufacturer Potential Power (SCADA) (kW)` is entirely empty in every 2021–2022 turbine file and should be excluded from the processed analytical schema unless future source data provide meaningful values.

`Investment Performance Ratio` and `Potential Power Energy Budget (kW)` have complete coverage in 2021–2022. They may be preserved as optional later-year fields, but their meaning and appropriate analytical use must remain distinct from the required cross-year performance measurements.

`Operating Performance Ratio` contains varying values but has only approximately 81.50–87.98% coverage. It should therefore remain an optional nullable later-year field rather than a required input.

The stable cross-year investigation fields remain the foundation of the cleaned SCADA table, while these schema additions are treated as year-specific supplementary evidence.

## 7. Final Cross-Year Data Contract

### 7.1 Clean SCADA table

The clean SCADA table will contain one row per turbine and ten-minute UTC timestamp. Its logical record key is `turbine_id` plus `timestamp`, while the source filename is retained for traceability.

Required cross-year source fields:

- `Wind speed (m/s)`
- `Power (kW)`
- `Potential power default PC (kW)`
- `Data Availability`
- `Available Capacity for Production (kW)`
- `Lost Production to Downtime (kWh)`

These columns are required in the schema, but their measurement values may remain missing. Missing measurements will be preserved and reported rather than filled automatically.

Optional supporting fields:

- `Lost Production to Performance (kWh)` as a signed vendor metric
- `Lost Production to Curtailment (Total) (kWh)` as year-sensitive evidence
- `Investment Performance Ratio` for 2021–2022
- `Operating Performance Ratio` for 2021–2022
- `Potential Power Energy Budget (kW)` for 2021–2022
- static turbine context such as rated power and hub height

`Manufacturer Potential Power (SCADA) (kW)` is excluded because it is entirely empty in every file where it appears.

Derived investigation fields will include power gap, ten-minute actual and potential energy, power-gap energy, capacity state, and explicit data-quality flags.

The cleaning pipeline will preserve raw files, negative actual-power values, signed performance-loss values, small power measurements above the 2,050 kW nominal rating, and the missing first two days of 2016. It will not apply blanket imputation, create artificial measurements, or automatically clip valid observations.

### 7.2 Clean Status-event table

The clean Status table will contain one row per unique Status event after removing redundant exact copies. Each record will receive a derived event identifier and retain its turbine identifier, source filename, and source row information for traceability.

Required fields:

- turbine identifier
- parsed start timestamp
- record type
- `Status`
- `Code`
- `Message`
- source filename

Nullable interval fields:

- parsed end timestamp
- parsed duration

Optional classification fields:

- `IEC category`
- `Service contract category`
- `Custom contract category`
- `Global contract category`

Records marked with `-` for end time and duration will be preserved as start-only events. Closed intervals will require a valid start, end, and duration, with the end not preceding the start and the supplied duration matching the timestamp-derived duration.

The malformed 2016 Turbine 2 record will retain its valid start timestamp and descriptive maintenance evidence but will receive an invalid-interval flag. Its end timestamp and duration will not be used for overlap or duration calculations.

The cleaned table will retain one copy of each exact Status record while storing a duplicate count or quality flag. Overlapping but non-identical events will remain separate because multiple operational statuses can occur simultaneously.

Raw Status exports will remain unchanged, and missing category values will not be imputed.

## 8. Export the Public File Inventory

A metadata-only inventory is exported for reproducibility. It lists the expected raw filenames, turbine and year identifiers, file type, source folder, and approximate size without exposing local absolute paths or raw measurements.

In [42]:
public_inventory_columns = [
    "file_type",
    "file_name",
    "file_year",
    "turbine_id",
    "year_folder",
    "size_mb",
]

public_file_inventory = file_inventory[
    public_inventory_columns
].copy()

public_file_inventory["size_mb"] = public_file_inventory[
    "size_mb"
].round(3)

public_file_inventory = public_file_inventory.sort_values(["file_year", "file_type", "turbine_id"]).reset_index(drop=True)

inventory_output_path = (
    project_root / "data" / "file_inventory.csv"
)

public_file_inventory.to_csv(inventory_output_path, index=False)
print(
    "Inventory exported to:",
    inventory_output_path.relative_to(project_root),
)
print(public_file_inventory.shape)

Inventory exported to: data\file_inventory.csv
(84, 6)
